# Data Exploration & Feature Engineering Research Notebook
## Alice M1 Humanoid Robot — Pick-and-Place Success/Failure Analysis

**Role:** Data & Pipeline Lead  
**Purpose:** Document the complete exploration process, experiments, decisions, and handover for the team.

This notebook is designed to tell the story of the analysis:
1. Understand the raw episodes and data quality.
2. Explore robot movement and task phases.
3. Investigate grasp behaviour.
4. Build and compare engineered features.
5. Examine correlations and feature usefulness.
6. Run exploratory classification experiments.
7. Prepare fixed-length sequences for the LSTM workflow.
8. Record limitations and decisions for the team.

> **Important:** This is a research notebook, not a final model-evaluation report. Run it from the repository root. It expects the shared `src/` package and the raw episode folders to exist locally. Raw CSVs and the holdout data should not be committed to GitHub.

## 0. Research rules and dataset split

The project contains 48 labelled episodes: 27 success and 21 fail. The agreed holdout contains 4 success and 4 fail episodes. The remaining 40 episodes are the development set.

**Leakage rule:** use development episodes for exploration, feature development, and cross-validation. Do not use holdout episodes to select features, tune models, or make development decisions.

**Transparency note:** the team explored all 48 episodes before the holdout was formally locked. Therefore, the holdout is kept separate for the project's final evaluation, but it should not be described as completely unseen during all earlier exploration. Mention this limitation in the report.

### 0.1 Imports and project paths
The path helper below works whether the notebook kernel starts in the repository root or inside the `notebooks/` folder.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the repository root by locating the shared src/ folder.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "src").is_dir()), None)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_HOLDOUT = ROOT / "data" / "holdout"

print("Project root:", ROOT)
print("Raw data folder exists:", DATA_RAW.exists())
print("Holdout folder exists:", DATA_HOLDOUT.exists())

### 0.2 Import the shared pipeline

The notebook should use the same shared functions as the project demo. This avoids having one implementation in the notebook and a different implementation in `src/`.

In [ ]:
from src.load import load_folder, check_episodes
from src.phases import detect_phases
from src.features import extract_features
from src.sequences import build_sequences, to_fixed_length
from src.config import (
    FPS, HOLDOUT_IDS, LEFT_JOINTS, ARM_JOINTS,
    FINGER_JOINTS, THUMB_JOINTS, HAND_JOINTS,
    SEQ_LEN, state, action
)

print("Shared pipeline imports successful.")
print("FPS:", FPS, "| Sequence length:", SEQ_LEN)

## 1. Load episodes and create the development view

First inspect the available folder structure. The loader supports labelled folders (`success/`, `fail/`) and unlabeled episode folders. Adjust `RAW_FOLDER` only if your repository uses a different raw-data directory.

In [ ]:
RAW_FOLDER = DATA_RAW
episodes_all = load_folder(RAW_FOLDER)

print("Episodes loaded:", len(episodes_all))
print("Example episode IDs:", list(episodes_all)[:5])

### 1.1 Separate development and holdout episodes

`HOLDOUT_IDS` is stored in the shared config. We compare IDs as strings because filenames and numeric IDs may use different formatting.

In [ ]:
holdout_ids = {str(x).zfill(6) for x in HOLDOUT_IDS}

def normalise_episode_id(value):
    return str(value).replace("episode_", "").zfill(6)

all_ids_normalised = {normalise_episode_id(x) for x in episodes_all}
missing_holdout_ids = holdout_ids - all_ids_normalised

if missing_holdout_ids:
    print("Warning: holdout IDs not found in loaded raw data:", sorted(missing_holdout_ids))

dev_episodes = {
    episode_id: df
    for episode_id, df in episodes_all.items()
    if normalise_episode_id(episode_id) not in holdout_ids
}

holdout_episodes = {
    episode_id: df
    for episode_id, df in episodes_all.items()
    if normalise_episode_id(episode_id) in holdout_ids
}

print(f"All episodes: {len(episodes_all)}")
print(f"Development episodes: {len(dev_episodes)}")
print(f"Holdout episodes: {len(holdout_episodes)}")

**Interpretation:** the expected split is 40 development episodes and 8 holdout episodes. If the counts differ, stop and check the folder names, `HOLDOUT_IDS`, and loader output before proceeding.

In [ ]:
assert len(episodes_all) == 48, "Expected 48 labelled episodes; verify raw data."
assert len(dev_episodes) == 40, "Expected 40 development episodes."
assert len(holdout_episodes) == 8, "Expected 8 holdout episodes."

## 2. Dataset structure and data-quality checks

Each episode is a time series. The known schema contains 57 columns: 5 information columns, 26 `state.*` channels, and 26 `action.*` channels. Verify the actual files instead of assuming every CSV is perfect.

In [ ]:
first_id = next(iter(dev_episodes))
sample = dev_episodes[first_id]

print("Sample episode:", first_id)
print("Shape:", sample.shape)
display(sample.head())
display(sample.dtypes.value_counts())

### 2.1 Run the shared episode validation

In [ ]:
validation = check_episodes(dev_episodes)
display(validation if isinstance(validation, pd.DataFrame) else pd.DataFrame(validation))

### 2.2 Check missing values, duplicate rows, and timestamps

A missing value can break feature extraction. Duplicate rows or non-monotonic timestamps can also distort speed and duration calculations.

In [ ]:
quality_rows = []
for episode_id, df in dev_episodes.items():
    time_col = "timestamp" if "timestamp" in df.columns else None
    quality_rows.append({
        "episode": episode_id,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "timestamp_monotonic": (
            bool(df[time_col].is_monotonic_increasing) if time_col else None
        ),
    })

quality_df = pd.DataFrame(quality_rows).sort_values("episode")
display(quality_df.describe(include="all"))
display(quality_df.head())

**How to interpret:** zero missing cells and monotonic timestamps are reassuring, but they do not prove that the robot performed the task correctly. Data quality checks only establish that the recorded data is structurally usable.

## 3. Episode length and duration exploration

Episodes contain different numbers of frames. At 30 FPS, approximate duration is `number_of_frames / 30`. If the dataset contains a timestamp column, use its measured range as the more direct duration estimate.

In [ ]:
length_rows = []
for episode_id, df in dev_episodes.items():
    label = None
    if "label" in df.columns:
        label = df["label"].iloc[0]
    elif "success" in str(episode_id).lower():
        label = "success"
    elif "fail" in str(episode_id).lower():
        label = "fail"

    if "timestamp" in df.columns:
        duration = float(df["timestamp"].iloc[-1] - df["timestamp"].iloc[0])
    else:
        duration = (len(df) - 1) / FPS

    length_rows.append({
        "episode": episode_id,
        "label": label,
        "frames": len(df),
        "duration_sec": duration,
    })

length_df = pd.DataFrame(length_rows)
display(length_df.groupby("label")[["frames", "duration_sec"]].agg(["mean", "std", "min", "max"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, group in length_df.groupby("label"):
    axes[0].hist(group["frames"], alpha=0.6, label=str(label))
    axes[1].hist(group["duration_sec"], alpha=0.6, label=str(label))
axes[0].set_title("Episode length (frames)")
axes[1].set_title("Episode duration (seconds)")
for ax in axes:
    ax.set_xlabel("Value")
    ax.set_ylabel("Number of episodes")
    ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** earlier exploration found episode lengths of roughly 270–520 rows (about 9–17 seconds). Duration may help describe behaviour, but it is not by itself a reliable success/fail rule. It is also strongly related to phase times, so correlated timing features should be interpreted carefully.

## 4. Robot movement: left arm vs right arm

The pick-and-place action is primarily performed by the left arm in these recordings. The right arm appeared to move very little, and its hand did not show the same closing behaviour. This section checks that observation quantitatively.

In [ ]:
print("Configured left-side channels:", LEFT_JOINTS)
print("Configured arm joints:", ARM_JOINTS)

In [ ]:
def available_state_columns(df, joints):
    cols = state(joints)
    return [c for c in cols if c in df.columns]

movement_rows = []
for episode_id, df in dev_episodes.items():
    row = {"episode": episode_id}
    for side, joints in [
        ("left", [j for j in ARM_JOINTS if str(j).startswith("l_")]),
        ("right", [j for j in ARM_JOINTS if str(j).startswith("r_")]),
    ]:
        cols = available_state_columns(df, joints)
        if cols:
            x = df[cols].to_numpy(dtype=float)
            row[f"{side}_total_abs_change"] = float(np.abs(np.diff(x, axis=0)).sum())
            row[f"{side}_mean_abs_change"] = float(np.abs(np.diff(x, axis=0)).mean())
    movement_rows.append(row)

movement_df = pd.DataFrame(movement_rows)
display(movement_df.describe())

**Interpretation:** compare the left and right movement totals. A larger left-arm movement supports focusing feature engineering on the task-relevant arm. It does not mean the right arm is universally irrelevant; that conclusion is limited to this task and these recordings.

## 5. Task phase detection

We describe the task as five broad phases:

1. **Reach** — the arm moves toward the object.
2. **Grasp** — the fingers close around the object.
3. **Carry** — the object is transported.
4. **Release** — the fingers open/release the object.
5. **Return** — the arm moves back after release.

The shared `detect_phases()` function identifies key event indices from the recorded signals. Inspect several episodes before trusting automatically detected events.

In [ ]:
phase_rows = []
for episode_id, df in dev_episodes.items():
    result = detect_phases(df)
    phase_rows.append({"episode": episode_id, **result})

phase_df = pd.DataFrame(phase_rows)
display(phase_df.head())
display(phase_df.select_dtypes(include="number").describe())

### 5.1 Visualise detected grasp and release points

This plot uses a left finger channel when available. The vertical lines show the event indices returned by the shared detector.

In [ ]:
episode_to_plot = next(iter(dev_episodes))
df_plot = dev_episodes[episode_to_plot]
events = detect_phases(df_plot)

finger_candidates = [c for c in state(FINGER_JOINTS) if c in df_plot.columns]
if finger_candidates:
    finger_col = finger_candidates[0]
    plt.figure(figsize=(11, 4))
    plt.plot(df_plot.index, df_plot[finger_col], label=finger_col)
    for key, colour in [("grasp_idx", "green"), ("release_idx", "red"), ("open_idx", "purple")]:
        idx = events.get(key)
        if idx is not None:
            plt.axvline(idx, linestyle="--", label=f"{key}={idx}")
    plt.title(f"Finger signal and detected events — {episode_to_plot}")
    plt.xlabel("Frame")
    plt.ylabel("Recorded joint value")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No configured finger state channel found in this episode.")

**Interpretation:** check whether the detected grasp and release indices line up with visible changes in the finger signal. Event detection is a useful approximation, not a human-verified ground-truth annotation.

## 6. Grasp behaviour experiment

Earlier exploration found that the left fingers move from approximately 0 toward 1.57 during grasp, while the right hand does not show comparable closing. The configured threshold `GRIP_CLOSED` is used by the phase detector.

In [ ]:
from src.config import GRIP_CLOSED

grip_rows = []
for episode_id, df in dev_episodes.items():
    cols = [c for c in state(FINGER_JOINTS) if c in df.columns]
    if not cols:
        continue
    finger_values = df[cols].to_numpy(dtype=float)
    grip_rows.append({
        "episode": episode_id,
        "finger_min": float(np.nanmin(finger_values)),
        "finger_max": float(np.nanmax(finger_values)),
        "finger_mean": float(np.nanmean(finger_values)),
        "any_value_above_threshold": bool(np.nanmax(finger_values) >= GRIP_CLOSED),
    })

grip_df = pd.DataFrame(grip_rows)
display(grip_df.describe(include="all"))

**Decision:** `n_grasps` was one for every one of the 48 episodes. Since it has no variance, it cannot distinguish success from failure in this dataset. Keep it only if the team needs it as a descriptive process feature; do not expect it to improve a classifier.

## 7. Feature engineering: what each feature measures

The shared feature extractor produces 14 features. These are compact summaries of timing, movement, tracking accuracy, and task events. Run the feature extractor to verify the current code version and feature names.

In [ ]:
feature_rows = []
for episode_id, df in dev_episodes.items():
    row = extract_features(df)
    row["episode"] = episode_id
    feature_rows.append(row)

features_dev = pd.DataFrame(feature_rows).set_index("episode")
print("Development feature table:", features_dev.shape)
display(features_dev.head())

### 7.1 Feature registry

| Feature | Meaning | Why investigate it? |
|---|---|---|
| `duration` | Total episode duration | Overall task efficiency |
| `reach_time` | Time before grasp | Approach speed |
| `carry_time` | Time from grasp to release | Object transport duration |
| `return_time` | Time from opening/release to end | Return behaviour |
| `reach_ratio` | Reach time divided by total duration | Relative time allocation |
| `mean_speed` | Average movement speed summary | Overall movement pace |
| `max_speed` | Peak movement speed summary | Fastest movement |
| `carry_speed` | Movement speed during carry | Transport pace |
| `path_length` | Total movement distance summary | Amount of movement |
| `track_err` | Mean absolute state-action difference over left arm joints | Tracking discrepancy |
| `max_track_err` | Maximum absolute state-action difference | Largest instantaneous discrepancy |
| `n_grasps` | Number of detected grasp events | Task event count |
| `shoulder_roll_movement` | Total shoulder-roll movement during grasp-to-release | Arm adjustment during carry |
| `return_track_err` | Mean state-action discrepancy during return | Return-phase tracking discrepancy |

**Note:** `state` and `action` are different recorded signals. A larger difference is not automatically proof of a worse task outcome; interpret it alongside the task and controller behaviour.

### 7.2 Check feature completeness and label balance

Use `results/features.csv` generated by the project's `scripts/build_features.py` as the authoritative label source. This avoids guessing labels from episode ID order or filename patterns. The notebook filters that table down to development episodes only.

In [ ]:
# Use the repository's generated feature table as the authoritative label source.
# If it does not exist yet, run: python scripts/build_features.py
existing_features_path = ROOT / "results" / "features.csv"

if not existing_features_path.exists():
    raise FileNotFoundError(
        f"{existing_features_path} not found. From the repository root, run "
        "'python scripts/build_features.py' first, then rerun this cell."
    )

all_feature_rows = pd.read_csv(existing_features_path)
required_label_columns = {"episode", "label"}
if not required_label_columns.issubset(all_feature_rows.columns):
    raise ValueError("results/features.csv must contain 'episode' and 'label' columns.")

all_feature_rows["episode_key"] = all_feature_rows["episode"].map(normalise_episode_id)
dev_keys = {normalise_episode_id(x) for x in dev_episodes}
dev_label_map = (
    all_feature_rows[all_feature_rows["episode_key"].isin(dev_keys)]
    .drop_duplicates("episode_key")
    .set_index("episode_key")["label"]
    .to_dict()
)

features_dev["label"] = [
    dev_label_map.get(normalise_episode_id(episode_id), np.nan)
    for episode_id in features_dev.index
]

print("Missing development labels:", int(features_dev["label"].isna().sum()))
display(features_dev["label"].value_counts(dropna=False))

If labels are missing, stop here and attach the labels using the repository's authoritative episode-label mapping. Never guess labels from row order or from model predictions.

## 8. Success vs failure: compare feature distributions

Compare group means and standard deviations. The mean difference is a first look; overlapping distributions mean a feature is not a perfect rule.

In [ ]:
labelled_features = features_dev.dropna(subset=["label"]).copy()
labelled_features["label"] = labelled_features["label"].astype(int)

group_summary = labelled_features.groupby("label").agg(["mean", "std", "min", "max"])
display(group_summary)

In [ ]:
numeric_features = [
    c for c in labelled_features.select_dtypes(include="number").columns
    if c != "label"
]

fig, axes = plt.subplots(4, 4, figsize=(15, 13))
axes = axes.flatten()
for ax, feature in zip(axes, numeric_features):
    fail_values = labelled_features.loc[labelled_features.label == 0, feature].dropna()
    success_values = labelled_features.loc[labelled_features.label == 1, feature].dropna()
    ax.boxplot([fail_values, success_values], tick_labels=["Fail", "Success"])
    ax.set_title(feature, fontsize=9)
    ax.tick_params(axis="x", labelsize=8)
for ax in axes[len(numeric_features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Interpretation:** boxplots show the centre, spread, and overlap. A feature whose success and fail boxes differ may be useful, but overlap and the small sample size still matter.

## 9. Standardised mean difference (SMD)

SMD expresses the difference between success and fail means in pooled-standard-deviation units. It helps compare features measured on different scales.

\[
SMD = \frac{\bar{x}_{success}-\bar{x}_{fail}}{s_{pooled}}
\]

Positive SMD means the success mean is higher; negative means the fail mean is higher. Large absolute SMD is a descriptive signal, not proof of predictive usefulness.

In [ ]:
def smd(success, fail):
    success = pd.Series(success).dropna()
    fail = pd.Series(fail).dropna()
    n1, n0 = len(success), len(fail)
    if n1 < 2 or n0 < 2:
        return np.nan
    pooled_var = (
        (n1 - 1) * success.var(ddof=1) +
        (n0 - 1) * fail.var(ddof=1)
    ) / (n1 + n0 - 2)
    if pooled_var <= 0:
        return np.nan
    return (success.mean() - fail.mean()) / np.sqrt(pooled_var)

smd_rows = []
for feature in numeric_features:
    succ = labelled_features.loc[labelled_features.label == 1, feature]
    fail = labelled_features.loc[labelled_features.label == 0, feature]
    smd_rows.append({
        "feature": feature,
        "fail_mean": fail.mean(),
        "success_mean": succ.mean(),
        "SMD": smd(succ, fail),
    })

smd_df = pd.DataFrame(smd_rows).sort_values("SMD", key=lambda s: s.abs(), ascending=False)
display(smd_df)

**Previously observed development-set pattern (40 episodes):** `shoulder_roll_movement` showed a relatively large positive SMD (about 0.79), followed by `max_speed` (about 0.68), `max_track_err` (about 0.64), and `return_time` (about -0.59). The `n_grasps` SMD is undefined because all episodes had one grasp.

Re-run the cell above against the current repository data. Small differences can occur if the feature code or split changes.

## 10. Feature correlation

Correlation helps identify features that may contain overlapping information. Strongly correlated features can make a model harder to interpret and can make feature-selection results unstable.

In [ ]:
corr = labelled_features[numeric_features].corr(method="pearson")
display(corr.round(2))

plt.figure(figsize=(12, 9))
plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

**Previously observed:** `mean_speed` and `track_err` were very strongly correlated (about +0.98); `mean_speed` and `carry_speed` were also strongly correlated (about +0.93); `carry_time` and `carry_speed` were strongly negatively correlated (about -0.87).

**Decision:** avoid treating highly correlated features as independent evidence. Keep a feature only when it adds a useful interpretation or improves a carefully evaluated model.

## 11. Exploratory classification: logistic regression

This experiment asks whether a small set of engineered features can separate success and fail episodes. It is exploratory because there are only 40 development episodes.

We use a pipeline so that imputation and scaling are fitted separately inside each training fold. This avoids leaking validation-fold statistics into the training fold.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

model_data = labelled_features.dropna(subset=["label"]).copy()
y = model_data["label"].astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_feature_set(feature_names, X_source=model_data, y_source=y):
    X = X_source[feature_names]
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    scores = cross_val_score(pipe, X, y_source, cv=cv, scoring="accuracy")
    return scores.mean(), scores.std(), scores

print("Development samples:", len(model_data))
print("Label counts:", y.value_counts().to_dict())

### 11.1 Baseline feature set

The initial baseline used four features: `max_speed`, `max_track_err`, `return_time`, and `mean_speed`.

In [ ]:
baseline_features = ["max_speed", "max_track_err", "return_time", "mean_speed"]
baseline_result = evaluate_feature_set(baseline_features)
print(f"Baseline CV accuracy: {baseline_result[0]:.3f} ± {baseline_result[1]:.3f}")

Earlier run: baseline mean accuracy was approximately **0.625** across 5 stratified folds. This is a small-sample estimate, not a final test score.

### 11.2 Add shoulder-roll movement

We tested whether shoulder movement during carry adds information beyond the baseline.

In [ ]:
baseline_plus_shoulder = baseline_features + ["shoulder_roll_movement"]
result_plus_shoulder = evaluate_feature_set(baseline_plus_shoulder)
print(f"Baseline + shoulder roll: {result_plus_shoulder[0]:.3f} ± {result_plus_shoulder[1]:.3f}")

Earlier run: adding shoulder-roll movement to the four-feature baseline reduced mean CV accuracy from about 0.625 to about 0.575. This does not mean the feature is useless: it may behave differently in other combinations, but the result warns against assuming that a large SMD guarantees better model performance.

### 11.3 Feature-combination experiment

We previously tested 31 combinations from a small candidate pool. The best observed pair was `max_track_err` + `shoulder_roll_movement`, with mean CV accuracy around 0.700.

**Important selection-bias warning:** selecting the best combination using the same CV folds used to report its score makes the reported best score optimistic. Treat this as exploratory feature screening, not an unbiased estimate of generalisation.

In [ ]:
from itertools import combinations

candidate_features = [
    "max_speed",
    "max_track_err",
    "return_time",
    "mean_speed",
    "shoulder_roll_movement",
]

combination_results = []
for k in range(1, len(candidate_features) + 1):
    for combo in combinations(candidate_features, k):
        mean_score, std_score, _ = evaluate_feature_set(list(combo))
        combination_results.append({
            "features": " + ".join(combo),
            "n_features": len(combo),
            "mean_accuracy": mean_score,
            "std_accuracy": std_score,
        })

combination_df = pd.DataFrame(combination_results).sort_values(
    "mean_accuracy", ascending=False
)
display(combination_df.head(10))

### 11.4 Return tracking error experiment

We then investigated whether tracking discrepancy during the return phase was more informative than whole-episode tracking error.

Earlier results:
- `return_track_err` had an SMD of approximately **+0.82**.
- Its 5-fold CV accuracy was approximately **0.725 ± 0.05** when tested alone.
- The success and fail ranges still overlapped (approximately 0.0368–0.0671 for success and 0.0371–0.0649 for fail).

This is a promising exploratory feature, but overlapping values mean it is not a deterministic success/fail threshold.

In [ ]:
if "return_track_err" in model_data.columns:
    return_result = evaluate_feature_set(["return_track_err"])
    print(f"Return tracking error CV accuracy: {return_result[0]:.3f} ± {return_result[1]:.3f}")
    display(model_data.groupby("label")["return_track_err"].agg(["mean", "std", "min", "max"]))
else:
    print("return_track_err is not present. Check src/features.py and rebuild the feature table.")

### 11.5 Confusion matrix example

A confusion matrix shows which class the model gets wrong. The earlier exploratory model using `max_track_err` + `shoulder_roll_movement` produced:

| | Predicted fail | Predicted success |
|---|---:|---:|
| Actual fail | 10 | 7 |
| Actual success | 5 | 18 |

That is 28 correct out of 40 (70%). This result came from an exploratory evaluation and should not be presented as final holdout performance.

### 11.6 Why we did not keep every candidate feature

Earlier checks found:
- `n_grasps` had no variance because every episode had one grasp.
- An `early_min` shoulder-angle feature did not improve CV performance.
- Return movement magnitude across joints was nearly the same between classes, so we did not claim that successful episodes simply moved more during return.
- Testing all seven left-arm joints for return tracking error did not clearly justify reducing to only the top three; the smaller-joint model had more variable fold scores.

These are useful negative results. Documenting what did not help prevents the team from repeating the same experiments without a reason.

## 12. Fixed-length sequence preparation for LSTM

The LSTM workflow expects every episode to have the same number of time steps. `to_fixed_length()` resamples the 13 left-side state channels to 300 steps.

This normalises the relative timeline, but it removes absolute duration information. Keep duration as a separate feature if the model needs it.

In [ ]:
sequence_example = to_fixed_length(next(iter(dev_episodes.values())), n=SEQ_LEN)
print("One sequence shape:", sequence_example.shape)

In [ ]:
X_sequences, sequence_ids = build_sequences(dev_episodes, n=SEQ_LEN)
print("All development sequences:", X_sequences.shape)
print("Number of IDs:", len(sequence_ids))

**Interpretation:** expected shape is `(number_of_episodes, 300, 13)`. The first dimension is episodes, the second is resampled time steps, and the third is left-side state channels. The LSTM owner should create labels in exactly the same `sequence_ids` order.

## 13. Build and save the development feature table

The project's shared `features.csv` should be created from development episodes for model development. Keep holdout data separate. Check the team's data-commit policy before committing generated CSV outputs.

In [ ]:
# Keep only development episodes in the shared development feature output.
feature_table = features_dev.copy()
feature_table = feature_table.reset_index().rename(columns={"index": "episode"})

print("Feature table shape:", feature_table.shape)
display(feature_table.head())

In [ ]:
# Optional local output. This writes a derived development-only table.
# Confirm the repository's agreed output path and Git policy before committing it.
output_dir = ROOT / "results"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "features_dev.csv"
feature_table.to_csv(output_path, index=False)
print("Saved:", output_path)

## 14. Final findings and decisions

### Main findings from the exploration
1. The left arm is the task-relevant arm in the observed pick-and-place episodes.
2. The left finger signal provides a useful grasp/release cue.
3. Every episode had one detected grasp, so `n_grasps` does not separate the labels.
4. Timing and movement features differ between classes, but their distributions overlap.
5. Some features are strongly correlated, especially `mean_speed` and `track_err`.
6. `return_track_err` looked promising in exploratory analysis, but its value ranges overlap across labels.
7. Feature-combination CV results are unstable and subject to selection bias on a 40-episode development set.
8. Fixed-length sequences are ready for the LSTM workflow, with absolute duration retained separately if needed.

### Handover
- **Imran:** use development-only features/sequences for classifier experiments; do not use holdout for augmentation, tuning, or feature selection.
- **Elisya:** use the phase and feature summaries for clustering/visualisation; clearly distinguish exploratory visual patterns from validated conclusions.
- **Wafi:** review the shared feature schema, labels, evaluation protocol, and whether derived CSV outputs may be committed.
- **Akif:** maintain the loader, phase detection, feature extraction, sequence preparation, and notebook documentation.

## 15. Limitations and reproducibility checklist

Before using this notebook in the report or demo:

- [ ] Run all cells from top to bottom in a fresh kernel.
- [ ] Confirm all 48 episodes load and the split is 40 development / 8 holdout.
- [ ] Confirm labels are explicit and no rows have missing labels.
- [ ] Confirm all phase indices are within the episode bounds.
- [ ] Check plots for representative success and fail episodes.
- [ ] Confirm feature names match `src/features.py`.
- [ ] Ensure no holdout episodes enter feature selection or cross-validation.
- [ ] Record package versions and the Git commit hash used for the final run.
- [ ] Do not report exploratory CV accuracy as final test accuracy.
- [ ] Disclose that all 48 episodes were explored before the holdout was locked.
- [ ] Do not commit raw episode CSVs; follow the team's approval rule for generated result CSVs.

**Reproducibility cell:**

In [ ]:
import platform
import sklearn
import matplotlib

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

---
**End of notebook.** This notebook records the exploration path and the reasoning behind the feature pipeline. It should be rerun against the current repository before any final report figures or metrics are copied.